# Cafe checkout times: Dixon's test on the 118 seconds

**Outlier Test (Smirnov-Grubbs)** · a worked example from the StatNex teaching guide

**The story.** A cafe manager timed 20 customers at the till. One checkout took 118 seconds, much longer than any other.

**The question.** Is the 118-second checkout an outlier, and is any other time suspicious?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `01_basic/normal_probability_plot.csv` (20 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Cup weight (g),Checkout time (sec),Arrival minute,Spend per customer ($)
1.0,356.0,49.0,14.0,2.79
2.0,357.4,51.0,39.0,9.31
3.0,351.8,39.0,52.0,9.03
4.0,346.5,47.0,8.0,4.51
5.0,335.4,51.0,9.0,9.28
6.0,361.1,48.0,29.0,10.25
7.0,353.3,47.0,17.0,3.91
8.0,353.5,45.0,48.0,33.3
9.0,350.1,40.0,51.0,32.52
10.0,346.0,56.0,16.0,6.04
11.0,357.1,44.0,42.0,7.41
12.0,349.3,118.0,26.0,7.42
13.0,352.2,48.0,2.0,7.38
14.0,356.3,38.0,3.0,5.82
15.0,363.7,43.0,21.0,4.81
16.0,349.8,43.0,0.0,4.25
17.0,342.6,44.0,26.0,7.27
18.0,358.9,52.0,56.0,33.72
19.0,349.8,46.0,48.0,6.22
20.0,348.0,51.0,52.0,1.76

These cells save `dixon.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile dixon.py
"""Dixon's Q test (r10) for one outlier, for small samples (up to 30 values). SciPy and
statsmodels have no such test.

Q is the gap between the candidate and its nearest neighbour, over the range. Its P
value is exact for a normal sample rather than read from a printed table: the joint
density of the smallest, second-largest and largest order statistics, integrated over
the second-largest in closed form, leaves

    P(Q >= q) = n(n - 1) ∬ φ(a) φ(a + d) [Φ(a + d(1 - q)) - Φ(a)]^(n - 2) da dd

over the range d, which scipy integrates numerically. Dixon's (1950) and Rorabacher's
(1991) critical values are recovered to their three decimals. Under a two-sided test
the end with the larger Q is examined and the one-sided tail is doubled.
"""

import math

import numpy as np
from scipy import integrate, special


def dixon_tail(q, n):
    """P(Q >= q) for a normal sample of n values."""
    # φ(a)φ(a + d) written out, and Φ as scipy.special.ndtr: the same values as
    # stats.norm.pdf and .cdf, at about a hundredth of the cost per call. The integrand
    # is evaluated 40,000 to 85,000 times, so this is seconds against milliseconds.
    def density(a, d):
        return (math.exp(-0.5 * (a * a + (a + d) ** 2)) / (2 * math.pi)
                * (special.ndtr(a + d * (1 - q)) - special.ndtr(a)) ** (n - 2))

    # Relative accuracy only: a far tail's integral is itself below 1e-12.
    value, _ = integrate.dblquad(density, 0, 20, lambda d: -10, lambda d: 10,
                                 epsabs=0, epsrel=1e-10)
    return min(1.0, n * (n - 1) * value)


def dixon(x, side="two-sided"):
    """(position of the candidate in x, Q, P). side: "two-sided", "largest" or "smallest"."""
    order = np.argsort(x, kind="stable")
    span = x[order[-1]] - x[order[0]]
    if span == 0:
        return -1, 0.0, 1.0
    q_top = (x[order[-1]] - x[order[-2]]) / span
    q_bottom = (x[order[1]] - x[order[0]]) / span
    top = q_top >= q_bottom if side == "two-sided" else side == "largest"
    q, index = (q_top, int(order[-1])) if top else (q_bottom, int(order[0]))
    tail = dixon_tail(q, x.size)
    return index, q, min(1.0, 2 * tail if side == "two-sided" else tail)

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from dixon import dixon

pd.set_option("display.width", 140)

## The data

In [ ]:
# One column per variable. Blank and text cells are skipped but keep their row
# number, so a result can be traced back to its row.
# Every test below assumes the values other than the candidate come from one normal
# distribution. A small P says the candidate sits further out than such a sample
# would plausibly put its most extreme value; it cannot say why (a typo, a fault, a
# genuine rare case), so check the record before deleting anything, and report the
# results with and without it. Testing one value at a time can miss two outliers
# lying close together, which inflate the SD between them (masking).
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Checkout time (sec)"]
alpha = 5 / 100
# Two-sided: the candidate is whichever end, the largest or the smallest value, has
# the larger gap to its neighbour.
side = "two-sided"

## Repeated Dixon test

In [ ]:
# Test the candidate (dixon.py); if P < alpha, exclude it and test again on what is
# left, until a candidate is retained or fewer than 3 values remain. Dixon's test is
# meant for small samples, and the app refuses a column of more than 30 values. Rows
# are tracked by position, so of two equal values only the excluded one is marked.
results = {}
for name in columns:
    raw = pd.to_numeric(data[name], errors="coerce").to_numpy(float)
    alive = np.flatnonzero(~np.isnan(raw))
    total = alive.size
    removed, steps = [], []
    while alive.size >= 3:
        x = raw[alive]
        index, statistic, p = dixon(x, side)
        # When every value is the same, dixon() returns -1: nothing can be an outlier.
        if index < 0:
            break
        is_max = x[index] >= x.mean() if side == "two-sided" else side == "largest"
        rejected = p < alpha
        steps.append([len(steps) + 1, x.size, x.mean(), x.var(ddof=1), x.std(ddof=1),
                      np.nan if is_max else x.min(), x.max() if is_max else np.nan,
                      alive[index] + 1, p, "rejected" if rejected else "retained",
                      "largest is an outlier" if is_max else "smallest is an outlier"])
        if not rejected:
            break
        removed.append(alive[index])
        alive = np.delete(alive, index)

    outlier = np.isin(np.arange(raw.size), removed)
    results[name] = {
        "data": pd.DataFrame({"No.": np.arange(1, raw.size + 1), "original": raw,
                              "after exclusion": np.where(outlier, np.nan, raw),
                              "outlier": np.where(outlier, raw, np.nan)}),
        "summary": pd.Series({"original": total, "after exclusion": total - len(removed),
                              "outliers": len(removed)}),
        "trace": pd.DataFrame(steps, columns=["step", "n", "mean", "variance", "sd", "min",
                                              "max", "No.", "P", "verdict", "alternative"]),
    }
    print(f"\n=== {name} ===")
    print(results[name]["data"].to_string(index=False))
    print()
    print(results[name]["summary"].to_string())
    print()
    print(results[name]["trace"].to_string(index=False))

## What the result says

Dixon's test compares the gap between the most extreme value and its neighbours with the range of the data. An outlier is a value too extreme to belong with the rest.

In step 1 the largest time, 118 seconds, is rejected. Its P value, the chance of a gap this large if nothing were wrong, is practically zero. In step 2 the largest of the 19 remaining times is 56 seconds. It is retained with P = 0.303, so no other time is suspicious.

Without that one customer the mean (average) falls from 50 to about 46.4 seconds, and the standard deviation, the typical spread, falls from 16.7 to 4.7 seconds.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.